# Representation Learning

## Prerequisites

In [193]:
# install required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [194]:
# checking if the GPU is enabled
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

True


In [195]:
# using Google Drive to save checkpoints
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Making ResNet from scratch

In [196]:
# matches simCLR = done
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
# forward pass
    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out

In [197]:
class ResNet18(nn.Module):
    def __init__(self,zero_init_residual=False):
        super(ResNet18, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True) # added ReLU - different from simCLR

        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Linear(512, 512)

        self.projection_head = nn.Sequential(nn.Linear(512, 512),
                                             nn.ReLU(inplace=True),
                                             nn.Linear(512, 128))

        for m in self.modules():
          if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
              nn.init.constant_(m.bias, 0)

        if zero_init_residual:
          for m in self.modules():
            if isinstance(m, BasicBlock):
              nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        z = self.projection_head(out)
        return out, z

# return resnet18 model here !!!!
model = ResNet18().to(device)

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.39 GiB is allocated by PyTorch, and 47.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Transformations (for SimCLR)

In [ ]:
# generates two augmented views of the same image - for SimCLR contrastive learning
class SimCLRDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        x1 = self.transform(x)
        x2 = self.transform(x)
        return x1, x2

    def __len__(self):
        return len(self.dataset)

In [ ]:
# transformations - random crops and horizontal flips
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(size=32),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# no test set needed
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
dataset = SimCLRDataset(trainset, transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=True, num_workers=4, drop_last=True)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck') # 10 classes

In [ ]:
# only used during training
class ProjectionHead(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(512, 512)
    self.bn = nn.BatchNorm1d(512)
    self.relu = nn.ReLU(inplace=True)
    self.fc2 = nn.Linear(512, 128)

  def forward(self, x):
    x = self.fc1(x)
    x = self.bn(x)
    x = self.relu(x)
    x = self.fc2(x)
    return F.normalize(x, dim=1)

In [ ]:
# SimCLR - wrapping ResNet18 and the projection head
class SimCLR(nn.Module):
  def __init__(self):
    super().__init__()
    self.encoder = model
    self.projection_head = ProjectionHead()

  def forward(self, x1, x2): #two images
      z1 = self.encoder(x1)
      z2 = self.encoder(x2)

      p1 = self.projection_head(z1)
      p2 = self.projection_head(z2)

      return p1, p2

In [ ]:
# contrastive loss
def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.size(0)

    z = torch.cat([z1, z2], dim=0)  # (2N, 128)

    similarity = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)
    similarity /= temperature

    # extract positives from the FULL matrix before masking
    positives = torch.cat([
        torch.diag(similarity, N),
        torch.diag(similarity, -N)
    ], dim=0)  # (2N,)

    # mask self-similarity
    mask = torch.eye(2*N, dtype=torch.bool).to(z.device)
    similarity = similarity.masked_fill(mask, float('-inf'))

    logits = similarity  # (2N, 2N)
    labels = torch.cat([
        torch.arange(N, 2*N),  # first N samples match with last N
        torch.arange(0, N)     # last N samples match with first N
    ]).to(z.device)

    loss = F.cross_entropy(logits, labels)
    return loss

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.025, momentum=0.9, weight_decay=1e-4, nesterov=True)
#T_max should be num_epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=0)

In [ ]:
def reset_weights(m):
    if hasattr(m, 'reset_parameters'):
        m.reset_parameters()

In [ ]:
# for loading a checkpoint only..!
"""
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/F_Supervised/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1

print(f"Loaded checkpoint from epoch {start_epoch}, loss: {checkpoint['loss']:.4f}")
"""

In [ ]:
model = SimCLR().to(device)

In [ ]:
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/F_Supervised/models/simclr_latest.pth'
num_epochs = 500
best_loss = 1000 # just a high number

for epoch in range(num_epochs):
    model.apply(reset_weights) # reinitialising weights between iterations
    model.train()
    running_loss = 0
    for (x1, x2) in loader:
        x1 = x1.to(device)
        x2 = x2.to(device)

        # Forward pass
        _, z1 = model(x1, x2)   # z1 = projection head output (128-dim)
        _, z2 = model(x1, x2)

        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    avg_loss = running_loss / len(loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    # save the best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'loss': avg_loss},
                   checkpoint_path)
        print(f"Saved best model - epoch {epoch+1} with loss: {avg_loss:.4f}")

In [ ]:
# loading trained model once training has stopped
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/F_Supervised/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path)

model = SimCLR().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()